In [1]:
import pandas as pd

In [2]:
iris= pd.read_csv("data/iris.csv")

In [3]:
iris.head(2)

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa


In [5]:
X= iris.iloc[:,:-1]
y= iris.iloc[:, -1]

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
# split the data into train and test
X_train, X_test, y_train, y_test= train_test_split(X,y, test_size=0.25, random_state=42)

In [8]:
X_train.shape,   y_train.shape

((112, 4), (112,))

In [9]:
X_test.shape, y_test.shape

((38, 4), (38,))

In [61]:
from sklearn.preprocessing import LabelEncoder

In [62]:
# initialize label encoder
encoder= LabelEncoder()

In [63]:
# encode y_train
y_train_encoded= encoder.fit_transform(y_train)

In [64]:
# encode y_test as well
y_test_encoded= encoder.transform(y_test)

# Hyper Param Tuning using Combination of Random and GridSearchCV

In [65]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.tree import DecisionTreeClassifier

In [66]:
# initialize Decision Tree Classifier
classifier= DecisionTreeClassifier()

In [95]:
# defining decision tree classifier params in RandomSearchCV as param_dist_random
param_dist_random= {
    "max_depth": list(np.arange(3,7, 1)), # this will start with 3,4,5,6,7 as max depth of tree
    "min_samples_split": list(np.arange(2, 11, 1)), # the min samples to split the node. it will try 2,3,..11 
    "min_samples_leaf": list(np.arange(1, 5, 1))   # the min samples required to be a leaf. it will try 1,2,3..5
}

In [96]:
# Use RandomizedSearchCV to perform a random search
random_search = RandomizedSearchCV(
    classifier, # i have defined the above i,e labelEncoding and DecisionTreeClassifier together pipeline
    param_distributions=param_dist_random, 
    n_iter=10, 
    cv=5, 
    scoring='accuracy', 
    random_state=42
)

In [97]:
# Fit the random_search model to the train data
random_search.fit(X_train, y_train_encoded)

RandomizedSearchCV(cv=5, estimator=DecisionTreeClassifier(),
                   param_distributions={'max_depth': [3, 4, 5, 6],
                                        'min_samples_leaf': [1, 2, 3, 4],
                                        'min_samples_split': [2, 3, 4, 5, 6, 7,
                                                              8, 9, 10]},
                   random_state=42, scoring='accuracy')

In [98]:
# Get the best hyperparameters from Randomized Search
best_params_random = random_search.best_params_

In [99]:
best_params_random

{'min_samples_split': 4, 'min_samples_leaf': 3, 'max_depth': 4}

In [102]:
# Refine the hyperparameter grid based on the results from random search
refined_param_grid = {
    "max_depth":list(np.arange(best_params_random["max_depth"] - 2, 
                                        best_params_random["min_samples_split"] + 3, 1)
                    ),
    "min_samples_split": list(np.arange(best_params_random["min_samples_split"] - 2, 
                                        best_params_random["min_samples_split"] + 3, 1)
                             ),
    "min_samples_leaf": list(np.arange(best_params_random["min_samples_leaf"] - 1, 
                                       best_params_random["min_samples_leaf"] + 2, 1)
                            )
}

In [103]:
# Perform Grid Search around the best parameters from Randomized Search
random_grid_search = GridSearchCV(
        classifier, 
        param_grid= refined_param_grid,
        cv=5,       # Cross-validation folds
        scoring="accuracy",
        verbose=1
    )

In [104]:
random_grid_search.fit(X_train, y_train_encoded)

Fitting 5 folds for each of 75 candidates, totalling 375 fits


GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'max_depth': [2, 3, 4, 5, 6],
                         'min_samples_leaf': [2, 3, 4],
                         'min_samples_split': [2, 3, 4, 5, 6]},
             scoring='accuracy', verbose=1)

In [105]:
# Get the best hyperparameters from Grid Search
best_params_grid = random_grid_search.best_params_

In [106]:
best_params_grid

{'max_depth': 4, 'min_samples_leaf': 3, 'min_samples_split': 2}

In [107]:
# Compare the results
print("Best Hyperparameters from Randomized Search:", best_params_random)
print("Best Hyperparameters from Grid Search:", best_params_grid)
print("Best Accuracy Score: ", random_grid_search.best_score_)

Best Hyperparameters from Randomized Search: {'min_samples_split': 4, 'min_samples_leaf': 3, 'max_depth': 4}
Best Hyperparameters from Grid Search: {'max_depth': 4, 'min_samples_leaf': 3, 'min_samples_split': 2}
Best Accuracy Score:  0.9545454545454547


In [108]:
# Evaluate on the test set
best_model_grid = random_grid_search.best_estimator_
y_pred= best_model_grid.predict(X_test)

In [109]:
# lets get the metrics
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [110]:
accuracy = accuracy_score(y_test_encoded, y_pred)
conf_matrix= confusion_matrix(y_test_encoded, y_pred)
class_report= classification_report(y_test_encoded, y_pred)

In [111]:
accuracy

1.0

In [112]:
conf_matrix

array([[15,  0,  0],
       [ 0, 11,  0],
       [ 0,  0, 12]], dtype=int64)

In [113]:
print(class_report)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        15
           1       1.00      1.00      1.00        11
           2       1.00      1.00      1.00        12

    accuracy                           1.00        38
   macro avg       1.00      1.00      1.00        38
weighted avg       1.00      1.00      1.00        38

